# Módulo 4 · Referencias y el montaje bipolar TCP

**Objetivo:** entender qué es la *referencia* de un EEG, por qué TUSZ tiene tres tipos de
montaje (`01_tcp_ar`, `02_tcp_le`, `03_tcp_ar_a`) y construir el montaje **TCP de 22 canales**
en el que están hechas las anotaciones por canal. Este será el formato de entrada de tus redes.

### Teoría en 1 minuto
Un electrodo no mide un voltaje absoluto: mide la **diferencia** respecto a otro punto.

- **Referencial**: cada electrodo contra una referencia común.
  - `LE` (*linked ears*): contra las orejas unidas (A1+A2).
  - `AR` (*average reference*): contra la media de todos los electrodos.
- **Bipolar**: cada canal = electrodo A − electrodo B **vecino**. Es lo que mira el neurólogo
  en el día a día. El montaje **TCP** (Temporal Central Parasagittal) son 22 de estos pares,
  organizados en cadenas.

Lo bonito: **si tienes los canales referenciales, puedes construir cualquier bipolar** restando:
`(FP1 − ref) − (F7 − ref) = FP1 − F7`. La referencia se cancela. Por eso el TCP de un fichero
LE y el de uno AR son comparables, y por eso **conviene trabajar siempre en TCP** en el TFM.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tusz_utils import read_tusz_raw, find_edf, TCP_PAIRS, TCP_NAMES, tusz_annotations, set_1020

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")

raw = read_tusz_raw("aaaaabqv_s001_t000")      # LE, con anotaciones absz
raw.pick("eeg")
print(raw.ch_names)

## 4.1 Las 22 parejas del montaje TCP
(Definidas en `DOCS/` de TUSZ y copiadas en `tusz_utils.TCP_PAIRS`.)

In [ ]:
pd.DataFrame(TCP_PAIRS, columns=["ánodo", "cátodo"], index=range(1, 23)).T

Visualmente, sobre la cabeza:

In [ ]:
r_pos = set_1020(raw.copy())
pos = {ch: r_pos.info["chs"][i]["loc"][:2] for i, ch in enumerate(r_pos.ch_names)}
fig, ax = plt.subplots(figsize=(5.5, 5.5))
colors = ["tab:blue"] * 4 + ["tab:orange"] * 4 + ["tab:green"] * 6 + ["tab:purple"] * 8
for (a, c), col in zip(TCP_PAIRS, colors):
    (x1, y1), (x2, y2) = pos[a], pos[c]
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-", color=col, lw=2.5))
for ch, (x, y) in pos.items():
    ax.plot(x, y, "o", color="w", mec="k", ms=17, zorder=3)
    ax.text(x, y, ch, ha="center", va="center", fontsize=7, zorder=4)
ax.set_aspect("equal"); ax.axis("off"); ax.set_title("Montaje TCP: 4 cadenas, 22 canales")
plt.show()

## 4.2 Construirlo a mano con NumPy (para entenderlo)

In [ ]:
X = raw.get_data()                                  # (n_canales, n_muestras)
idx = {ch: i for i, ch in enumerate(raw.ch_names)}
X_tcp = np.stack([X[idx[a]] - X[idx[c]] for a, c in TCP_PAIRS])
print(X_tcp.shape)

info_tcp = mne.create_info(TCP_NAMES, sfreq=raw.info["sfreq"], ch_types="eeg")
info_tcp.set_meas_date(raw.info["meas_date"])   # misma fecha → las anotaciones encajan
raw_tcp_manual = mne.io.RawArray(X_tcp, info_tcp)
raw_tcp_manual.set_annotations(raw.annotations)
raw_tcp_manual

`mne.create_info` + `mne.io.RawArray` es la receta para **crear un Raw desde cualquier array**:
te servirá también para meter en MNE señales que generes tú (p. ej. reconstrucciones de un autoencoder).

> ¿Por qué `set_meas_date`? Al asignarse a un Raw, las anotaciones guardan como origen (`orig_time`)
> la fecha de inicio del registro. Si el Raw nuevo no tiene fecha, MNE no sabe cómo alinearlas y da error.
> Alternativa: crear las anotaciones con `orig_time=None` (segundos desde la primera muestra).

## 4.3 La forma MNE: `mne.set_bipolar_reference`

In [ ]:
raw_tcp = mne.set_bipolar_reference(raw.copy(), anode=[a for a, _ in TCP_PAIRS],
                                    cathode=[c for _, c in TCP_PAIRS], drop_refs=True)
raw_tcp.pick(TCP_NAMES)           # quita los canales sobrantes (FZ, PZ, OZ, T1, T2) y fija el orden
print(raw_tcp.ch_names)

In [ ]:
# ¿Coinciden las dos versiones?
print("Máx. diferencia:", np.abs(raw_tcp.get_data() - raw_tcp_manual.get_data()).max(), "V")

Todo esto está en `tusz_utils.make_tcp(raw)`, que además se adapta a ficheros a los que les faltan
A1/A2 (montaje `03_tcp_ar_a` → 20 canales).

In [ ]:
fig = raw_tcp.plot(start=176, duration=15, n_channels=22, scalings=dict(eeg=200e-6),
                   highpass=1, lowpass=30, show_scrollbars=False,
                   title="Montaje TCP · crisis de ausencia")

Compara con la vista referencial del módulo 2: en bipolar la descarga generalizada se ve
**más pequeña** en algunos canales, porque los dos electrodos de cada par oscilan casi a la vez
y la resta la cancela parcialmente. Es una característica, no un error: el bipolar resalta
diferencias **locales** (ideal para crisis focales) y atenúa lo difuso.

## 4.4 Referencia promedio (CAR)

Otra opción habitual en ML: re-referenciar a la media (*common average reference*).
`set_eeg_reference` lo hace en una línea:

In [ ]:
raw_car = raw.copy().pick(["FP1", "FP2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
                           "F7", "F8", "T3", "T4", "T5", "T6", "FZ", "CZ", "PZ"])
raw_car.set_eeg_reference("average")
print(raw_car.info["custom_ref_applied"])

> **¿TCP o CAR para el TFM?** Recomendación: **TCP**. (1) Las anotaciones por canal de TUSZ están en TCP,
> (2) unifica los tres tipos de montaje de TUSZ y (3) es lo que usa la mayoría de la literatura sobre
> TUSZ, así tus resultados serán comparables.

## 4.5 Anotaciones por canal

El `.csv` de TUSZ dice **en qué canales** se ve cada crisis. MNE soporta anotaciones con
`ch_names`, y como ya tenemos canales con los mismos nombres (`FP1-F7`…), podemos usarlas.
En crisis focales (`fnsz`, `cpsz`) esto te dice **dónde** está el foco: oro para la parte de XAI.

In [ ]:
ann_ch = tusz_annotations(find_edf("aaaaabqv_s001_t000"), kind="multi", per_channel=True)
print(ann_ch)
print(ann_ch.ch_names[:5])

In [ ]:
# Matriz canal × crisis: ¿qué canales participan en cada crisis?
df = pd.DataFrame({"canal": [c[0] for c in ann_ch.ch_names], "onset": ann_ch.onset.round(0)})
pd.crosstab(df.canal, df.onset).reindex(TCP_NAMES).fillna(0).astype(int).T

En la ausencia aparecen **los 22 canales** en las 3 crisis: generalizada.

---
## 🧪 Ejercicios

**E4.1** Construye el TCP del fichero `aaaaamtc_s001_t000` (montaje **AR**) con `make_tcp`
y comprueba que tiene los mismos 22 canales y en el mismo orden que el de `aaaaabqv`.

**E4.2** Busca en el inventario (`seizures.csv` / `files.csv`) un fichero con crisis **focal** (`fnsz`)
que tengas descargado, carga sus anotaciones por canal y mira qué canales participan.
¿Qué región de la cabeza es?

**E4.3** Monta un montaje bipolar **longitudinal alternativo** (la "doble banana" sin la cadena central:
solo cadenas temporales y parasagitales = 16 canales) con `set_bipolar_reference`.

### Solución E4.1

In [ ]:
from tusz_utils import make_tcp
tcp_ar = make_tcp(read_tusz_raw("aaaaamtc_s001_t000"))
print(tcp_ar.ch_names == raw_tcp.ch_names, len(tcp_ar.ch_names))

### Solución E4.2

In [ ]:
from tusz_utils import INVENTORY
files = pd.read_csv(INVENTORY / "files.csv")
focal = files[files.labels.fillna("").str.fullmatch("fnsz")].sort_values("edf_dur_s")
print(focal[["file", "montage", "n_seiz_events_bi", "seiz_secs_bi"]].head())
fid = None
for f in focal.file:                      # el primero que esté en disco
    try:
        edf_f = find_edf(f); fid = f; break
    except FileNotFoundError:
        pass
if fid:
    a = tusz_annotations(edf_f, per_channel=True)
    print(fid, pd.Series([c[0] for c in a.ch_names]).value_counts())
else:
    print("Ningún fichero fnsz descargado en este equipo")

### Solución E4.3

In [ ]:
pairs16 = TCP_PAIRS[:8] + TCP_PAIRS[14:]
banana = mne.set_bipolar_reference(raw.copy(), [a for a, _ in pairs16], [c for _, c in pairs16])
banana.pick([f"{a}-{c}" for a, c in pairs16])
print(len(banana.ch_names), banana.ch_names)